# Setup
Notebooks call reusable functions from `src/`.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config, resolve_paths, set_global_seed, get_seed
config = load_config(ROOT / 'configs/project_config.yaml')
paths = resolve_paths(config)
set_global_seed(get_seed(config))
print('project root:', paths.root)


## Label construction

In [ ]:
from src.data_loader import load_parquet
from src.temporal_split import chronological_date_split, masks_from_split
from src.label_engineering import construct_all_labels, label_distribution
feat = load_parquet(paths.processed).sort_values('timestamp').reset_index(drop=True)
split = chronological_date_split(feat, config['split']['train_fraction'], config['split']['validation_fraction'], config['split']['test_fraction'], config['split']['purge_seconds'])
masks = masks_from_split(len(feat), split)
labeled, eps = construct_all_labels(feat, config, __import__('pandas').Series(masks['train'], index=feat.index))
print('epsilons', eps)
print(label_distribution(labeled['label']))
